In [1]:
from ERA_Distribution_Classes_Python.Classes.ERADist import ERADist
from ERA_Distribution_Classes_Python.Classes.ERANataf import ERANataf
from ERA_Distribution_Classes_Python.Classes.FORM_HLRF import FORM_HLRF
from ERA_Distribution_Classes_Python.Classes.FORM_fmincon import FORM_fmincon
from ERA_Distribution_Classes_Python.Classes.SuS import SuS

In [2]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt 
from structure import Structure
from solver_2nd import Solver2ndOrder
from measures_of_nonlinearity import kappa_1, kappa_2, kappa_12, r1, r2
from syst_4_model_functions import t_S_nonlinear

## Material Properties

In [7]:
# intermediate modified IPE 120 -> eta = 100% for linear hyperplane
E = 210e6 # kN/m2 
A = 1.321e-3 # m2
I = 2.7784576742780014e-06 # m4
h = 0.12  # m
z = h/2  # m
alpha = 1.14

In [8]:
def t_R(M_k, I=I, z=z, alpha=alpha):
    """
    Takes in the Steel Bending Strength M_k (random variable) in kN/cm2
    I in m^4
    z in m
    alpha is the plastic ratio 

    Returns the characteristic Bending Moment Resistance M_c_Rk for the given system in kNm
    """
    return I/z * M_k * 100e2 * alpha

## System Definition

In [9]:
# Vectorized Version of the Structural response function (Better for array handling later)
t_S_vectorized = np.vectorize(t_S_nonlinear, otypes=[float])

## Target characteristic values for calibrating random variables

In [11]:
s_k = 1.1 # snow load on ground kN/m2
q_b = 0.65 # wind pressure kN/m2
w_k = q_b * 0.8 # wind load kN/m2 with c_pe,10 = 0.8 (Area D)
m_k = 35.5

In [12]:
# Quick Check
print(t_S_nonlinear(l_1=s_k * 1.5, l_2=q_b * 1.5))

21.13299229197635


In [8]:
# Design Opt 1
print(t_S_nonlinear(l_1= s_k*1.5, l_2= q_b*1.5) * 1.0 / t_R(m_k))

# Design Opt 2
e_d = max(1.5 * t_S_nonlinear(l_1= s_k, l_2= q_b), 1.5 * t_S_nonlinear(l_1= s_k, l_2= q_b))
print(e_d * 1.0 / t_R(m_k))

1.1403198044097296
1.0923155730833936


## t_S_nonlinear results for calibrating t_S_hyperplane_linear

In [9]:
print(f"t_S_(l_1k, 0) = {t_S_nonlinear(l_1=s_k, l_2=0.0)}")
print(f"t_S_(0, l_2k) = {t_S_nonlinear(l_1=0.0, l_2=q_b)}")

t_S_(l_1k, 0) = 1.521903900412326
t_S_(0, l_2k) = 10.971906842505982


## Random Variables

In [10]:
# Snow time-invariant part
mu_Theta_1 = 0.81
cov_Theta_1 = 0.26
sig_Theta_1 = mu_Theta_1 * cov_Theta_1
Theta_L1 = ERADist('lognormal','MOM',[mu_Theta_1, sig_Theta_1])

# Snow load on ground
mu_L1 = 1.0
cov_L1 = 0.2
sig_L1 = mu_L1 * cov_L1
L1 = ERADist('gumbel','MOM',[mu_L1,sig_L1])

In [11]:
percentile_L1 = L1.icdf(0.98)
print(f"Snow 98% Percentile: {percentile_L1}")

Snow 98% Percentile: 1.5184551765313796


In [12]:
# Wind time-invariant part
mu_Theta_2 = 0.97
cov_Theta_2 = 0.26
sig_Theta_2 = mu_Theta_2 * cov_Theta_2
Theta_L2 = ERADist('lognormal','MOM',[mu_Theta_2, sig_Theta_2])

# Wind velocity pressure
mu_L2 = 1.0 
cov_L2 = 0.14
sig_L2 = mu_L2 * cov_L2
L2 = ERADist('gumbel','MOM',[mu_L2, sig_L2])

In [13]:
percentile_L2 = L2.icdf(0.98)
print(f"Wind 98% Percentile: {percentile_L2}")

Wind 98% Percentile: 1.3629186235719657


In [14]:
# Structural Response Model Uncertainty (from JCSS Probabilistic Model Code, Part 3, Table 3.9.1)
mu_Theta_S = 1.0
cov_Theta_S = 0.1
sig_Theta_S = mu_Theta_S * cov_Theta_S
Theta_S = ERADist('lognormal','MOM',[mu_Theta_S, sig_Theta_S])  # Distribution for Moments in frames

In [15]:
# Steel bending model uncertainty
mu_Theta_M = 1.15
cov_Theta_M = 0.05
sig_Theta_M = mu_Theta_M * cov_Theta_M
Theta_M = ERADist('lognormal','MOM',[mu_Theta_M, sig_Theta_M])

# Steel yielding strength
mu_M = 1.0
cov_M = 0.05
sig_M = mu_M * cov_M
M = ERADist('lognormal','MOM',[mu_M, sig_M])

In [16]:
percentile_M = M.icdf(0.05)
print(f"Steel 5% Percentile: {percentile_M}")

Steel 5% Percentile: 0.9199464756612658


## Shifting / Scaling Random Variables

In [17]:
# Snow Load on Ground, shifted to characteristic value
snow_shift = s_k / percentile_L1 # ratio of target to current percentile, by which mean and std get multiplied

mu_L1_shifted = mu_L1 * snow_shift
sig_L1_shifted = sig_L1 * snow_shift
L1_shifted = ERADist('gumbel','MOM',[mu_L1_shifted, sig_L1_shifted])

print(f"""Snow Load on Ground gets shifted by {snow_shift}""")
print(f"""Old mean: {mu_L1}; New mean: {mu_L1_shifted}""")
print(f"""Old std: {sig_L1}; New std: {sig_L1_shifted}""")
print(f"""Old 98th percentile: {L1.icdf(.98)}; New 98th percentile: {L1_shifted.icdf(.98)}""")
print(f"""Old COV: {L1.std()/L1.mean()}; New COV: {L1_shifted.std()/L1_shifted.mean()}""")

Snow Load on Ground gets shifted by 0.7244204616646898
Old mean: 1.0; New mean: 0.7244204616646898
Old std: 0.2; New std: 0.14488409233293795
Old 98th percentile: 1.5184551765313796; New 98th percentile: 1.0999999999999999
Old COV: 0.19999999999999998; New COV: 0.19999999999999996


In [18]:
# Wind velocity pressure, shifted to characteristic value
wind_shift = q_b / percentile_L2

mu_L2_shifted = mu_L2 * wind_shift
sig_L2_shifted = sig_L2 * wind_shift
L2_shifted = ERADist('gumbel','MOM',[mu_L2_shifted, sig_L2_shifted])

print(f"""Wind velocity pressure gets shifted by {wind_shift}""")
print(f"""Old mean: {mu_L2}; New mean: {mu_L2_shifted}""")
print(f"""Old std: {sig_L2}; New std: {sig_L2_shifted}""")
print(f"""Old 98th percentile: {L2.icdf(.98)}; New 98th percentile: {L2_shifted.icdf(.98)}""")
print(f"""Old COV: {L2.std()/L2.mean()}; New COV: {L2_shifted.std()/L2_shifted.mean()}""")

Wind velocity pressure gets shifted by 0.4769176888173018
Old mean: 1.0; New mean: 0.4769176888173018
Old std: 0.14; New std: 0.06676847643442226
Old 98th percentile: 1.3629186235719657; New 98th percentile: 0.65
Old COV: 0.14; New COV: 0.13999999999999999


In [19]:
# Steel bending resistance, shifted to characteristic value
steel_shift = m_k / percentile_M

mu_M_shifted = mu_M * steel_shift
sig_M_shifted = sig_M * steel_shift
M_shifted = ERADist('lognormal','MOM',[mu_M_shifted, sig_M_shifted])

print(f"""Steel bending resistance gets shifted by {steel_shift}""")
print(f"""Old mean: {mu_M}; New mean: {mu_M_shifted}""")
print(f"""Old std: {sig_M}; New std: {sig_M_shifted}""")
print(f"""Old 5th percentile: {M.icdf(0.05)}; New 5th percentile: {M_shifted.icdf(0.05)}""")
print(f"""Old COV: {M.std()/M.mean()}; New COV: {M_shifted.std()/M_shifted.mean()}""")

Steel bending resistance gets shifted by 38.58920158858403
Old mean: 1.0; New mean: 38.58920158858403
Old std: 0.05; New std: 1.9294600794292016
Old 5th percentile: 0.9199464756612658; New 5th percentile: 35.5
Old COV: 0.04999999999999947; New COV: 0.04999999999999946


### Distribution Plots

In [20]:
x_plotting = np.linspace(0, 2.0, 200) # for PDF plotting only
x_resistance_plotting = np.linspace(30, 50, 200) # for PDF plotting only

In [21]:
# # FIGURE 1: LOADS Side by side
# # ================================================================================
# fig1, axes = plt.subplots(1, 2, figsize=(16, 5))

# # --- Left Plot: SNOW ---
# axes[0].plot(x_plotting, Theta_L1.pdf(x_plotting), label=fr'$\Theta_1$ (Lognormal, $\mu={Theta_L1.mean():.2f}, COV={Theta_L1.std()/Theta_L1.mean():.2f}$)')
# axes[0].plot(x_plotting, L1.pdf(x_plotting), label=fr'$Q_1$ (Gumbel, $\mu={L1.mean():.2f}, COV={L1.std()/L1.mean():.2f}$)')
# axes[0].plot(x_plotting, L1_shifted.pdf(x_plotting), label=fr'$Q_1,shifted$ (Gumbel, $\mu={L1_shifted.mean():.2f}, COV={L1_shifted.std()/L1_shifted.mean():.2f}$)', linestyle= "--", color = "orange")
# axes[0].set_title('Snow Load Components')
# axes[0].set_xlabel('$x$')
# axes[0].set_ylabel('$f(x)$')
# axes[0].grid(True, linestyle='--', alpha=0.6)
# axes[0].legend(fontsize="medium")
# axes[0].set_ylim(top=axes[0].get_ylim()[1] * 1.25)

# # --- Right Plot: WIND ---
# axes[1].plot(x_plotting, Theta_L2.pdf(x_plotting), label=fr'$\Theta_2$ (Lognormal, $\mu={Theta_L2.mean():.2f}, COV={Theta_L2.std()/Theta_L2.mean():.2f}$)')
# axes[1].plot(x_plotting, L2.pdf(x_plotting), label=fr'$Q_2$ (Gumbel, $\mu={L2.mean():.2f}, COV={L2.std()/L2.mean():.2f}$)')
# axes[1].plot(x_plotting, L2_shifted.pdf(x_plotting), label=fr'$Q_2,shifted$ (Gumbel, $\mu={L2_shifted.mean():.2f}, COV={L2_shifted.std()/L2_shifted.mean():.2f}$)', linestyle= "--", color = "orange")
# axes[1].set_title('Wind Load Components')
# axes[1].set_xlabel('$x$')
# axes[1].set_ylabel('$f(x)$')
# axes[1].grid(True, linestyle='--', alpha=0.6)
# axes[1].legend(fontsize="medium")
# axes[1].set_ylim(top=axes[1].get_ylim()[1] * 1.25)

# fig1.tight_layout()
# plt.show()


In [22]:
# # FIGURE 2: Resistance
# # ================================================================================
# fig2, axes = plt.subplots(1, 2, figsize=(16, 5))

# # --- Left Plot: Resistance Original ---
# axes[0].plot(x_plotting, Theta_M.pdf(x_plotting), label=fr'$\Theta_M$ (Lognormal, $\mu={Theta_M.mean():.2f}, COV={Theta_M.std()/Theta_M.mean():.2f}$)', color='blue')
# axes[0].plot(x_plotting, M.pdf(x_plotting), label=fr'$M$ (Lognormal, $\mu={M.mean():.2f}, COV={M.std()/M.mean():.2f}$)', color='orange')
# axes[0].set_title('Steel Yielding Strength (Resistance)')
# axes[0].set_xlabel('$x$')
# axes[0].set_ylabel('$f(x)$')
# axes[0].grid(True, linestyle='--', alpha=0.6)
# axes[0].legend(fontsize="medium")
# axes[0].set_ylim(top=axes[0].get_ylim()[1] * 1.25)

# # --- Right Plot: Resistance Shifted (without Model uncertainty) --
# axes[1].plot(x_resistance_plotting, M_shifted.pdf(x_resistance_plotting), label=fr'$M,shifted$ (Lognormal, $\mu={M_shifted.mean():.2f}, COV={M_shifted.std()/M_shifted.mean():.2f}$)', color='orange', linestyle= "--")
# axes[1].set_title('Steel Yielding Strength (Resistance)')
# axes[1].set_xlabel('$x$')
# axes[1].set_ylabel('$f(x)$')
# axes[1].grid(True, linestyle='--', alpha=0.6)
# axes[1].legend(fontsize="medium")
# axes[1].set_ylim(top=axes[1].get_ylim()[1] * 1.25)

# fig1.tight_layout()
# plt.show()

## Characteristic values derived from Random Variables for further calculation

In [23]:
l_1k = L1_shifted.icdf(0.98)
l_2k = L2_shifted.icdf(0.98)
m_k = M_shifted.icdf(0.05)
print(f"l_1k = {l_1k:.4f} kN/m^2")
print(f"l_2k = {l_2k:.4f} kN/m^2")
print(f"m_k = {m_k:.4f} kN/cm^2")

l_1k = 1.1000 kN/m^2
l_2k = 0.6500 kN/m^2
m_k = 35.5000 kN/cm^2


## Partial Safety Factors

In [24]:
gamma_M = 1.0  # Resistance
gamma_F1 = 1.5   # Snow Load
gamma_F2 = 1.5   # Wind Load
#psi_0 = 1.0 # 0.6     # Windload

## Design Values

In [25]:
l_1d = s_k * gamma_F1
l_2d = q_b * gamma_F2 
m_d = m_k / gamma_M
print(f"l_1d = {l_1d:.4f} kN/m^2")
print(f"l_2d = {l_2d:.4f} kN/m^2")
print(f"m_d = {m_d:.4f} kN/cm^2")

l_1d = 1.6500 kN/m^2
l_2d = 0.9750 kN/m^2
m_d = 35.5000 kN/cm^2


## Measures of Nonlinearity

In [26]:
## Measures of Nonlinearity
k1 = kappa_1(l_1k=l_1k, l_1d=l_1d, t_S=t_S_nonlinear)
k2 = kappa_2(l_2k=l_2k, l_2d=l_2d, t_S=t_S_nonlinear)
k12 = kappa_12(l_1k=l_1k, l_1d=l_1d, l_2k=l_2k, l_2d=l_2d, t_S=t_S_nonlinear)
r1 = r1(l_1k=l_1k, l_2k=l_2k, t_S=t_S_nonlinear)
r2 = r2(l_1k=l_1k, l_2k=l_2k, t_S=t_S_nonlinear)


print(f"kappa1 = {k1}")
print(f"kappa2 = {k2}")
print(f"kappa12 = {k12}")
print(f"r1 = {r1}")
print(f"r2 = {r2}")

kappa1 = 1.6556150935902205
kappa2 = 0.9497325520989162
kappa12 = 1.1318416559534055
r1 = 0.11277063763196107
r2 = 0.8130006962536702


## Design parameters p for option 1 and 2 

In [27]:
# Design Opt 1
e_d_1 = t_S_nonlinear(l_1d, l_2d)

# Design Opt 2
argument_1 = gamma_F1 * t_S_nonlinear(l_1k,  (gamma_F2 / gamma_F1) * l_2k)
argument_2 = gamma_F2 * t_S_nonlinear((gamma_F1 / gamma_F2) * l_1k, l_2k)
e_d_2 = max(argument_1, argument_2)

p_opt1 = gamma_M * e_d_1 / t_R(M_k=m_d)
p_opt2 = gamma_M * e_d_2 / t_R(M_k=m_d)

print("Design Option 1:")
print(f"e_d = {e_d_1} kNm")
print(f"p_opt1 = {p_opt1}")
print(f"\n")
print("Design Option 2:")
print(f"e_d = {e_d_2} kNm")
print(f"p_opt2 = {p_opt2}")

Design Option 1:
e_d = 21.13299229197635 MPa
p_opt1 = 1.1403198044097296


Design Option 2:
e_d = 20.24335322170971 MPa
p_opt2 = 1.0923155730833936


## Construction of the Nataf Distribution

In [28]:
# Array of marginal distributions
marginal_dist = [Theta_M, M_shifted, Theta_L1, L1_shifted, Theta_L2, L2_shifted, Theta_S]

# Correlation matrix (no correlation yet)
dimensions = len(marginal_dist)
R_xx = np.eye(dimensions)

# Construction of the Nataf Distribution
nataf = ERANataf(M=marginal_dist, Correlation=R_xx)

## Subset Simulation

### Design Option 1

In [29]:
# deterministic design action effect
e_d_opt1 = t_S_nonlinear(l_1=gamma_F1 * s_k, l_2=gamma_F2 * q_b) # kNm

In [30]:
def g_opt_1_sus(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k) # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt1 / r_k) * x[:,0] * t_R(M_k=x[:,1])
    action_side = x[:,6] * t_S_vectorized(l_1=(x[:,2] * x[:,3]), l_2=(x[:,4] * x[:,5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [31]:
# # %% Samples Return
# samples_return = 1
# # %% subset simulation
# N  = 10000        # Total number of samples for each level
# p0 = 0.1         # Probability of each subset, chosen adaptively

# print('\n\nSUBSET SIMULATION: ')
# [Pf_SuS_1, delta_SuS, b, Pf, b_sus, pf_sus, samplesU, samplesX, fs_iid] = SuS(N, p0, g_opt_1_sus, nataf, samples_return)

In [32]:
# print("Subset Simulation for Design Option 1")
# print(f"P(F) = {Pf_SuS_1}")
# X = sp.stats.Normal()
# beta = - X.icdf(Pf_SuS_1)
# print(f"beta = {beta}")
# print(samplesX)

---

### Design Option 2

In [33]:
# deterministic design action effect
argument_1 = gamma_F1 * t_S_nonlinear(l_1= s_k, l_2= (gamma_F2 / gamma_F1) * q_b)
argument_2 = gamma_F2 * t_S_nonlinear(l_1= (gamma_F1 / gamma_F2) * s_k, l_2= q_b)
e_d_opt2 = max(argument_1, argument_2) # kNm

In [34]:
def g_opt_2_sus(x):
    """
    LSF for Design Option 2
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt2 / r_k) * x[:,0] * t_R(M_k=x[:,1])
    action_side = x[:,6] * t_S_vectorized(l_1=(x[:,2] * x[:,3]), l_2=(x[:,4] * x[:,5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [35]:
# # %% Samples Return
# samples_return = 1
# # %% subset simulation
# N  = 10000        # Total number of samples for each level
# p0 = 0.1         # Probability of each subset, chosen adaptively

# print('\n\nSUBSET SIMULATION: ')
# [Pf_SuS_2, delta_SuS, b, Pf, b_sus, pf_sus, samplesU, samplesX, fs_iid] = SuS(N, p0, g_opt_2_sus, nataf, samples_return)

In [36]:
# print("Subset Simulation for Design Option 2")
# print(f"P(F) = {Pf_SuS_2}")

# X = sp.stats.Normal()
# beta = - X.icdf(Pf_SuS_2)
# print(f"beta = {beta}")
# # print(samplesX)

## FORM Analysis

### Design Option 1

In [37]:
def g_opt_1_FORM(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k) # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt1 / r_k) * x[0] * t_R(M_k=x[1])
    action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [38]:
# Perform FORM with HLRF
# [u_star, x_star, beta, Pf, S_F1, S_F1_T] = FORM_HLRF(g=g_opt_1, dg=[], distr=nataf, sensitivity_analysis=0, u0=0)

# Perform FORM with fmincom
[u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_1_FORM, dg=[], distr=nataf, u0=0)


*scipy.optimize.minimize() with  SLSQP  Method

  20  iterations... Reliability index =  3.788347685138807  --- Failure probability =  7.582626095786595e-05 




In [39]:
print(f"u_star = {u_star}")
print(f"x_star = {x_star}")
print(f"(alpha_2)^2 = {(u_star/beta)**2}")
print(f"beta = {beta}")
print(f"P(F) = {Pf}")
print(f"g(X*) = {g_opt_1_FORM(x_star)}")

u_star = [-0.59964417 -0.59964425  0.25864208  0.18449363  2.75007392  2.12896343
  1.19837654]
x_star = [ 1.11466065 37.40336025  0.83754748  0.72541392  1.89684678  0.65969953
  1.12138498]
(alpha_2)^2 = [0.02505461 0.02505461 0.00466121 0.00237172 0.52697386 0.3158179
 0.10006609]
beta = 3.788347685138807
P(F) = 7.582626095786595e-05
g(X*) = -3.5464402152740604e-06


### Design Option 2

In [40]:
def g_opt_2_FORM(x):
    """
    LSF for Design Option 2
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt2 / r_k) * x[0] * t_R(M_k=x[1])
    action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [41]:
# Perform FORM with HLRF
# [u_star, x_star, beta, Pf, S_F1, S_F1_T] = FORM_HLRF(g=g_opt_2, dg=[], distr=nataf, sensitivity_analysis=0, u0=1)

# Perform FORM with fmincom
[u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_2_FORM, dg=[], distr=nataf)


*scipy.optimize.minimize() with  SLSQP  Method

  23  iterations... Reliability index =  3.6529822514521175  --- Failure probability =  0.00012960607723981623 




In [42]:
print(f"u_star = {u_star}")
print(f"x_star = {x_star}")
print(f"(alpha_2)^2 = {(u_star/beta)**2}")
print(f"beta = {beta}")
print(f"P(F) = {Pf}")
print(f"g(X*) = {g_opt_2_FORM(x_star)}")

u_star = [-0.57720588 -0.57720573  0.25689357  0.18514928  2.66792941  2.03243635
  1.15283425]
x_star = [ 1.11591112 37.44532138  0.83717301  0.725505    1.85741112  0.64729654
  1.11630218]
(alpha_2)^2 = [0.024967   0.02496699 0.00494551 0.00256891 0.53340066 0.30955569
 0.09959525]
beta = 3.6529822514521175
P(F) = 0.00012960607723981623
g(X*) = 5.111354184350603e-08


## Optimization of additional PSF $\gamma_{new}$

Remark: due to long simulation time with SuS, only FORM is used here

In [43]:
from scipy.optimize import minimize
from scipy.optimize import brentq

In [44]:
beta_TRG = 3.4451629852993326 # linear FORM solution, copied from syst_4_reliability_analysis_lin.ipynb

### Design Option (1) 

In [45]:
def f(gamma_new):
    def g_opt_1(x):
        """
        LSF for Design Option 1
        Input variables: 
        x[0]: Theta_M = Resistance Model Uncertainty
        x[1]: M = Steel yield strength
        x[2]: Theta_L1 = Snow Load Model Uncertainty
        x[3]: L1 = Snow Load on Ground
        x[4]: Theta_L2 = Wind Load Model Uncertainty
        x[5]: L2 = Wind velocity pressure
        x[6]: Theta_S = Structural Response Model Uncertainty
        """
        
        # deterministic characteristic resistance 
        r_k = t_R(M_k=m_k) # kNm
        
        # assembly of the LSF
        resistance_side = (gamma_M * gamma_new *e_d_opt1 / r_k) * x[0] * t_R(M_k=x[1])
        action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
        
        # print(f"resistance = {resistance_side}")
        # print(f"action = {action_side}")
        
        return resistance_side - action_side

    [u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_1, dg=[], distr=nataf)
    return beta - beta_TRG

In [48]:
# this is for narrowing down the lower and upper bound of where brentq should search in the input space
for g in [0.89, 0.90]:
    print(g, f(g))


*scipy.optimize.minimize() with  SLSQP  Method

  19  iterations... Reliability index =  3.420545680615278  --- Failure probability =  0.0003124781949541422 


0.89 -0.024617304684054542

*scipy.optimize.minimize() with  SLSQP  Method

  26  iterations... Reliability index =  3.455983246395189  --- Failure probability =  0.00027414469927948104 


0.9 0.01082026109585632


Observation: root lies between $x = 0.89$ and $x = 0.90$


In [49]:
# Finding the Root (Optimization Problem)
gamma_new_1 = brentq(f, 0.89, 0.90)


*scipy.optimize.minimize() with  SLSQP  Method

  19  iterations... Reliability index =  3.420545680615278  --- Failure probability =  0.0003124781949541422 



*scipy.optimize.minimize() with  SLSQP  Method

  26  iterations... Reliability index =  3.455983246395189  --- Failure probability =  0.00027414469927948104 



*scipy.optimize.minimize() with  SLSQP  Method

  18  iterations... Reliability index =  3.4451929341588516  --- Failure probability =  0.00028532596067852876 



*scipy.optimize.minimize() with  SLSQP  Method

  24  iterations... Reliability index =  3.445158598688715  --- Failure probability =  0.00028536220844084276 



*scipy.optimize.minimize() with  SLSQP  Method

  26  iterations... Reliability index =  3.4451662087106274  --- Failure probability =  0.00028535417421374166 



*scipy.optimize.minimize() with  SLSQP  Method

  24  iterations... Reliability index =  3.4451727341865226  --- Failure probability =  0.000285347285156215 



*scipy.optimize.minimize() 

In [50]:
print(f"gamma_new = {gamma_new_1}")

gamma_new = 0.8969382075957193


### Verification of $\gamma_{new}$ for Design Option (1)

In [51]:
def g_opt_1_verification(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k) # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * gamma_new_1 * e_d_opt1 / r_k) * x[0] * t_R(M_k=x[1])
    action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [52]:
# Perform FORM with fmincom
[u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_1_verification, dg=[], distr=nataf, u0=0)


*scipy.optimize.minimize() with  SLSQP  Method

  20  iterations... Reliability index =  3.445164346207863  --- Failure probability =  0.0002853561405184654 




In [53]:
print(f"Target Reliability Index: {beta_TRG}")
print(f"Optimized Reliability Index: {beta}")
print(f"Difference: {beta - beta_TRG}")

Target Reliability Index: 3.4451629852993326
Optimized Reliability Index: 3.445164346207863
Difference: 1.3609085303123436e-06


### Design Option (2) 

In [54]:
def f(gamma_new):
    def g_opt_2(x):
        """
        LSF for Design Option 2
        Input variables: 
        x[0]: Theta_M = Resistance Model Uncertainty
        x[1]: M = Steel yield strength
        x[2]: Theta_L1 = Snow Load Model Uncertainty
        x[3]: L1 = Snow Load on Ground
        x[4]: Theta_L2 = Wind Load Model Uncertainty
        x[5]: L2 = Wind velocity pressure
        x[6]: Theta_S = Structural Response Model Uncertainty
        """
        
        # deterministic characteristic resistance 
        r_k = t_R(M_k=m_k)  # kNm
        
        # assembly of the LSF
        resistance_side = (gamma_M * gamma_new * e_d_opt2 / r_k) * x[0] * t_R(M_k=x[1])
        action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
        
        # print(f"resistance = {resistance_side}")
        # print(f"action = {action_side}")
        
        return resistance_side - action_side

    [u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_2, dg=[], distr=nataf)
    return beta - beta_TRG

In [55]:
# this is for narrowing down the lower and upper bound of where brentq should search in the input space
for g in [0.91, 0.92, 0.93, 0.94]:
    print(g, f(g))


*scipy.optimize.minimize() with  SLSQP  Method

  22  iterations... Reliability index =  3.3545798140979066  --- Failure probability =  0.000397428096986921 


0.91 -0.090583171201426

*scipy.optimize.minimize() with  SLSQP  Method

  22  iterations... Reliability index =  3.3892837041309427  --- Failure probability =  0.00035037734116403964 


0.92 -0.055879281168389916

*scipy.optimize.minimize() with  SLSQP  Method

  25  iterations... Reliability index =  3.4236146705597315  --- Failure probability =  0.0003089708718988855 


0.93 -0.02154831473960117

*scipy.optimize.minimize() with  SLSQP  Method

  18  iterations... Reliability index =  3.4574930779592155  --- Failure probability =  0.00027261312442538205 


0.94 0.01233009265988283


Observation: root lies between $x = 0.93$ and $x = 0.94$.

In [56]:
# Finding the Root (Optimization Problem)
# increased the tolerance, otherwise solver will test out infeasible high loads and crash
gamma_new_2 = brentq(f, a=0.93, b=0.94 ,xtol=1e-4, rtol=1e-6, maxiter=50)


*scipy.optimize.minimize() with  SLSQP  Method

  25  iterations... Reliability index =  3.4236146705597315  --- Failure probability =  0.0003089708718988855 



*scipy.optimize.minimize() with  SLSQP  Method

  18  iterations... Reliability index =  3.4574930779592155  --- Failure probability =  0.00027261312442538205 



*scipy.optimize.minimize() with  SLSQP  Method

  17  iterations... Reliability index =  3.445184336851586  --- Failure probability =  0.0002853350364049587 



*scipy.optimize.minimize() with  SLSQP  Method

  26  iterations... Reliability index =  3.445011386897556  --- Failure probability =  0.00028551766770926196 




In [57]:
print(f"gamma_new = {gamma_new_2}")

gamma_new = 0.9363604863373622


### Verification of $\gamma_{new}$ for Design Option (2)

In [58]:
def g_opt_2_verification(x):
    """
    LSF for Design Option 2
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * gamma_new_2 * e_d_opt2 / r_k) * x[0] * t_R(M_k=x[1])
    action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [59]:
# Perform FORM with fmincom
[u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_2_verification, dg=[], distr=nataf, u0=0)


*scipy.optimize.minimize() with  SLSQP  Method

  20  iterations... Reliability index =  3.4452652974492493  --- Failure probability =  0.0002852495812003358 




In [60]:
print(f"Target Reliability Index: {beta_TRG}")
print(f"Optimized Reliability Index: {beta}")
print(f"Difference: {beta - beta_TRG}")

Target Reliability Index: 3.4451629852993326
Optimized Reliability Index: 3.4452652974492493
Difference: 0.00010231214991662796
